# Rank SAE units by C2C-label AUC

This notebook reproduces the basic feature-ranking idea: use training images to rank individual SAE units by how well their activation separates C2C-positive from C2C-negative images, then report AUC on the held-out test split. It also lets you restrict the analysis to one prompt group to reduce prompt-group confounding.

It uses cached pooled activations and C2C labels from the companion `debiasing` workspace. No model generation or SAE encoding is needed for this notebook.

In [ ]:
from pathlib import Path
import csv
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from IPython.display import display, clear_output

NOTEBOOK_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                     if p.name == "sae-demo-notebooks" and (p / "sae-activation").exists())
DEBIASING_ROOT = NOTEBOOK_ROOT.parent  # Edit this if debiasing is elsewhere.
FEATURES_PATH = DEBIASING_ROOT / "backprop/sae_features.npz"
TARGETS_PATH = DEBIASING_ROOT / "sae-training/c2c_targets.csv"


## Choose the analysis cohort

The maximum pool asks whether a unit activates strongly anywhere in an image. The mean pool asks whether its average activation is high. Features are selected using train AUC only; test AUC is shown afterward for those same selected feature IDs.

In [ ]:
concept = widgets.Dropdown(options=["flower", "kimono"], value="flower", description="Concept:")
group = widgets.Dropdown(options=["All prompts", "east asian woman"], value="east asian woman", description="Group:")
pooling = widgets.Dropdown(options=["maximum", "mean"], value="maximum", description="Pooling:")
top_k = widgets.IntSlider(value=10, min=1, max=30, step=1, description="Top features:")
button = widgets.Button(description="Rank features", button_style="primary")
output = widgets.Output()

feature_key = {"maximum": "X_max", "mean": "X_mean"}
label_key = {"flower": "flower_present", "kimono": "kimono_present"}


In [ ]:
def rank_features(_):
    with output:
        clear_output(wait=True)
        try:
            data = np.load(FEATURES_PATH, allow_pickle=False)
            ids = data["sample_id"].astype(str)
            X = data[feature_key[pooling.value]]
            with TARGETS_PATH.open(newline="") as f:
                labels_by_id = {r["sample_id"]: r for r in csv.DictReader(f)}
            rows = [labels_by_id.get(sid) for sid in ids]
            keep = np.array([r is not None and r["split"] in ("train", "test") and
                             (group.value == "All prompts" or
                              (r["race_prompt"] == "east asian" and r["gender_prompt"] == "woman"))
                             for r in rows])
            indices = np.flatnonzero(keep)
            labels = np.array([int(rows[i][label_key[concept.value]]) for i in indices])
            split = np.array([rows[i]["split"] for i in indices])
            train_idx, test_idx = split == "train", split == "test"
            if len(np.unique(labels[train_idx])) < 2 or len(np.unique(labels[test_idx])) < 2:
                raise ValueError("This cohort needs positive and negative examples in both train and test.")
            scores = X[indices]
            train_auc = np.array([roc_auc_score(labels[train_idx], scores[train_idx, j])
                                  for j in range(scores.shape[1])])
            order = np.argsort(-train_auc, kind="stable")
            selected = [int(j) for j in order if train_auc[j] > 0.5][:top_k.value]
            if not selected:
                selected = [int(order[0])]
            test_auc = [roc_auc_score(labels[test_idx], scores[test_idx, j]) for j in selected]
            print(f"Cohort: {group.value}; concept: {concept.value}; pooling: {pooling.value}")
            print(f"Train n={train_idx.sum()} (positive={labels[train_idx].sum()}), "
                  f"test n={test_idx.sum()} (positive={labels[test_idx].sum()})")
            print("Feature IDs are ranked on train data; test scores are held out.")
            for rank, (fid, auc_test) in enumerate(zip(selected, test_auc), 1):
                print(f"{rank:2d}. feature {fid:4d} | train AUC {train_auc[fid]:.3f} | test AUC {auc_test:.3f}")
            fig, ax = plt.subplots(figsize=(10, 4))
            x = np.arange(len(selected))
            ax.bar(x-0.18, [train_auc[j] for j in selected], width=0.36, label="Train AUC")
            ax.bar(x+0.18, test_auc, width=0.36, label="Test AUC")
            ax.axhline(0.5, color="black", linestyle="--", linewidth=1)
            ax.set_xticks(x, [str(j) for j in selected], rotation=45)
            ax.set_xlabel("SAE feature ID"); ax.set_ylabel("AUC"); ax.set_ylim(0, 1)
            ax.legend(); ax.set_title(f"{concept.value.title()} feature AUC")
            plt.show()
        except Exception as exc:
            print(f"Could not rank features: {type(exc).__name__}: {exc}")

button.on_click(rank_features)
display(concept, group, pooling, top_k, button, output)


An AUC above 0.5 means a unit tends to be more active on positive-labeled images. This is an association score, not proof that the unit represents the object or that suppressing it will remove the object. C2C labels are automatic working targets; small or imbalanced cohorts can make AUC unstable.